# 02｜Damodaran 行業估值數據庫
**資料來源：Aswath Damodaran（NYU Stern 商學院）— 估值領域最知名的學者**

> **核心問題：本益比 30 倍，是貴還是便宜？**
>
> 答案：要看跟誰比。本 Notebook 使用 Damodaran 整理的美股各產業估值資料，
> 建立「不同產業應該有不同估值基準」的分析框架。

## 「目標價 520 元」這個數字是怎麼來的？

每次大公司召開法說會，隔天各大券商研究報告就出來了：
A 說目標價 560，B 說 490，C 說 420。

你有沒有想過，這些數字是怎麼算出來的？

是猜的嗎？不完全是。背後有一套邏輯叫做**相對估值**：
找到同產業的公司，看看市場給它們什麼樣的本益比（P/E）、EV/EBITDA，
再把同樣的倍數套到你要估值的公司。

**問題是：這個「同產業平均」，從哪裡來？**

NYU Stern 商學院的 Damodaran 教授每年免費更新一份全球行業估值資料庫。
半導體、金融、零售、能源——幾百個行業的 P/E、毛利率、資本結構……全都在裡面。

這一章，我們直接拿這份數據，學會怎麼判斷一個行業的估值是「貴」還是「便宜」。

## 🎯 學習目標
1. 認識 Damodaran 每年更新的全球行業估值數據庫
2. 比較各行業 P/E、P/B、PEG、EV/EBITDA 的差異與成因
3. 建立「行業估值基準」，避免用同一把尺衡量所有股票

## ⚙️ Step 0：安裝套件

In [ ]:
%pip install openpyxl xlrd
print("✅ 完成")

## 匯入套件

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

matplotlib.rcParams['font.family'] = ['Microsoft YaHei', 'SimHei', 'Heiti TC', 'PingFang HK', 'STHeiti', 'Arial Unicode MS', 'sans-serif']
matplotlib.rcParams['axes.unicode_minus'] = False
print("✅ 匯入完成")

## 一、為什麼估值要和同業比？

### Gordon Growth Model（股利折現模型）
> **合理 P/E = 配息率 ÷ （必要報酬率 − 成長率）**

這個公式說明了決定 P/E 倍數的三個關鍵：

| 變數 | 效果 | 例子 |
|------|------|------|
| **成長率 ↑** | P/E 應該更高 | 科技股高成長 → 高 P/E 合理 |
| **必要報酬率（風險）↑** | P/E 應該更低 | 能源股風險高 → P/E 較低 |
| **配息率 ↑** | P/E 應該更高 | 公用事業穩定配息 → 較高 P/E |

**結論：不同產業天生就應該有不同的 P/E 基準，用同一把尺比較是錯誤的。**

## 二、載入 Damodaran 資料

In [ ]:
import os

LOCAL = "data/damodaran_pe.csv"

if os.path.exists(LOCAL):
    df = pd.read_csv(LOCAL)
    print(f"✅ 從本機載入（{len(df)} 個產業）")
else:
    print("⏳ 從 NYU Damodaran 網站下載...")
    url = "https://pages.stern.nyu.edu/~adamodar/pc/datasets/pedata.xls"

    # Damodaran 檔案有 7 行 metadata，第 8 行才是欄位標題
    try:
        raw = pd.read_excel(url, sheet_name='Industry Averages', skiprows=7, header=0)
    except Exception:
        raw = pd.read_excel(url, sheet_name='Industry Averages', skiprows=7, header=0, engine='xlrd')

    # 欄位位置（固定，不隨年份異動）：
    # 0=Industry Name, 1=N firms, 2=Loss%, 3=Current PE, 4=Trailing PE,
    # 5=Forward PE, 6=MktCap/NI(all), 7=MktCap/NI(profitable), 8=Growth 5y, 9=PEG
    df = pd.DataFrame()
    df['industry']  = raw.iloc[:, 0].astype(str).str.strip()
    df['n_firms']   = pd.to_numeric(raw.iloc[:, 1], errors='coerce')
    df['pe']        = pd.to_numeric(raw.iloc[:, 3], errors='coerce')  # Current PE
    df['pe_adj']    = pd.to_numeric(raw.iloc[:, 7], errors='coerce')  # Mkt Cap/NI (profitable firms)
    df['growth_5y'] = pd.to_numeric(raw.iloc[:, 8], errors='coerce') * 100  # Expected 5yr growth → %
    df['peg']       = pd.to_numeric(raw.iloc[:, 9], errors='coerce')  # PEG Ratio

    # 過濾無效資料
    df = df[df['industry'].str.len() > 2].copy()
    df = df[df['n_firms'].notna() & (df['n_firms'] > 5)].copy()
    df = df.dropna(subset=['pe_adj'])
    df = df[df['pe_adj'] > 0].copy()       # 移除負 P/E
    df = df[df['pe_adj'] < 500].copy()     # 移除異常高值

    df.to_csv(LOCAL, index=False)
    print(f"✅ 下載完成，已快取（{len(df)} 個產業）")

print(f"\n有效產業數：{len(df)}")
df.sort_values('pe_adj', ascending=False).head(10)

## 三、各產業 P/E 分布

In [ ]:
# 排序，取前後各 20 個產業，讓圖不要太擁擠
df_sorted = df.sort_values('pe_adj', ascending=True).reset_index(drop=True)

n_show = min(40, len(df_sorted))
df_plot = pd.concat([
    df_sorted.head(n_show // 2),
    df_sorted.tail(n_show // 2)
]).drop_duplicates()

colors = ['#e74c3c' if pe > df_sorted['pe_adj'].median() else '#2ecc71'
          for pe in df_plot['pe_adj']]

fig, ax = plt.subplots(figsize=(11, max(10, len(df_plot) * 0.32)))
bars = ax.barh(range(len(df_plot)), df_plot['pe_adj'], color=colors, alpha=0.8, edgecolor='white')
ax.set_yticks(range(len(df_plot)))
ax.set_yticklabels(df_plot['industry'], fontsize=8)
ax.axvline(df_sorted['pe_adj'].median(), color='black', linestyle='--', linewidth=1.2,
           label=f"中位數 ({df_sorted['pe_adj'].median():.1f}x)")
ax.set_xlabel('P/E（倍，排除虧損公司後調整）')
ax.set_title('美股各產業本益比（Damodaran）', fontsize=13)
ax.legend()
plt.tight_layout()
plt.show()

print(f"所有產業 P/E 中位數：{df['pe_adj'].median():.1f} 倍")
print(f"所有產業 P/E 均值：{df['pe_adj'].mean():.1f} 倍")

## 四、最貴 vs 最便宜的產業

In [ ]:
top10 = df.nlargest(10, 'pe_adj')[['industry','pe_adj','n_firms']].reset_index(drop=True)
bot10 = df.nsmallest(10, 'pe_adj')[['industry','pe_adj','n_firms']].reset_index(drop=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].barh(range(10), top10['pe_adj'][::-1], color='#e74c3c', alpha=0.8)
axes[0].set_yticks(range(10))
axes[0].set_yticklabels(top10['industry'][::-1], fontsize=9)
axes[0].set_xlabel('P/E（倍）')
axes[0].set_title('P/E 最高（最貴）10 個產業', fontsize=11)
for i, v in enumerate(top10['pe_adj'][::-1]):
    axes[0].text(v + 0.5, i, f'{v:.0f}x', va='center', fontsize=9)

axes[1].barh(range(10), bot10['pe_adj'], color='#2ecc71', alpha=0.8)
axes[1].set_yticks(range(10))
axes[1].set_yticklabels(bot10['industry'], fontsize=9)
axes[1].set_xlabel('P/E（倍）')
axes[1].set_title('P/E 最低（最便宜）10 個產業', fontsize=11)
for i, v in enumerate(bot10['pe_adj']):
    axes[1].text(v + 0.1, i, f'{v:.0f}x', va='center', fontsize=9)

plt.tight_layout()
plt.show()

## 五、是什麼決定了估值倍數？

**驗證：成長率越高的產業，P/E 是否真的越高？**

如果 Gordon Model 是對的，預期成長率高的產業應該獲得更高的 P/E 倍數。

In [ ]:
valid = df.dropna(subset=['pe_adj', 'growth_5y']).copy()
valid = valid[(valid['growth_5y'] > -50) & (valid['growth_5y'] < 100)]

fig, ax = plt.subplots(figsize=(9, 6))
sc = ax.scatter(valid['growth_5y'], valid['pe_adj'],
                s=valid['n_firms'].clip(5, 500) / 5,
                alpha=0.5, color='steelblue', edgecolors='none')

# 趨勢線
z = np.polyfit(valid['growth_5y'], valid['pe_adj'], 1)
p = np.poly1d(z)
xr = np.linspace(valid['growth_5y'].min(), valid['growth_5y'].max(), 100)
ax.plot(xr, p(xr), 'r-', linewidth=2, label='趨勢線')

corr = valid['growth_5y'].corr(valid['pe_adj'])
ax.set_title(f'預期成長率 vs P/E\n（相關係數 = {corr:.2f}，點的大小 = 公司數）', fontsize=12)
ax.set_xlabel('預期 5 年盈餘成長率（%）')
ax.set_ylabel('P/E（倍）')
ax.legend()
plt.tight_layout()
plt.show()

print(f"成長率 vs P/E 相關係數：{corr:.2f}")

## 六、PEG 比率 — 把成長率納入估值

In [ ]:
# PEG = P/E ÷ 成長率，理論上 PEG < 1 代表被低估
valid_peg = df.dropna(subset=['peg']).copy()
valid_peg = valid_peg[(valid_peg['peg'] > 0) & (valid_peg['peg'] < 10)]
valid_peg = valid_peg.sort_values('peg')

fig, ax = plt.subplots(figsize=(11, max(8, len(valid_peg) * 0.18)))
colors_peg = ['#2ecc71' if peg < 1.5 else '#e67e22' if peg < 3 else '#e74c3c'
              for peg in valid_peg['peg']]
ax.barh(range(len(valid_peg)), valid_peg['peg'], color=colors_peg, alpha=0.8)
ax.set_yticks(range(len(valid_peg)))
ax.set_yticklabels(valid_peg['industry'], fontsize=7)
ax.axvline(1.5, color='orange', linestyle='--', linewidth=1, label='PEG = 1.5（合理基準）')
ax.set_xlabel('PEG 比率')
ax.set_title('美股各產業 PEG 比率（綠 < 1.5 < 橙 < 3 < 紅）', fontsize=11)
ax.legend()
plt.tight_layout()
plt.show()

## 七、討論

**結論：估值必須在同業框架下比較，不同產業有天然不同的 P/E 基準。**

**思考問題：**
1. 軟體業的 P/E 通常遠高於銀行業。根據 Gordon Model，這合理嗎？原因是什麼？
2. PEG < 1 的產業是「真的便宜」還是「成長預期太低」？
3. 台積電的 P/E 應該跟哪些公司比？台灣半導體業的合理估值倍數是多少？
4. 熊市時估值會均值回歸（往均值靠攏）。哪些產業在熊市中 P/E 壓縮幅度最大？

**延伸應用：**
- 把 FinMind 的台股財報資料和這個框架結合，建立台股版的產業估值基準
- 每年 Damodaran 會更新資料，可以比較今年和去年的估值變化

## 八、這跟你有什麼關係？

**下次看到一支股票的 P/E，先問：「它跟誰比？」**

你剛才下載的 Damodaran 資料，就是你的估值基準表。
台灣學生最常討論的股票（台積電、聯發科、輝達）都在半導體產業——
讓我們直接查出這個產業的 P/E 基準是多少。

In [ ]:
# 查詢你感興趣的產業 P/E 基準
keywords = ['Semiconductor', 'Software', 'Bank', 'Pharma', 'Retail', 'Auto']

print("Damodaran 產業 P/E 基準（可用來對照個股估值）：")
print()
print(f"{'產業':^34} | {'公司數':>6} | {'現值P/E':>9} | {'預期成長':>8} | {'PEG':>6}")
print("-" * 72)
for kw in keywords:
    matches = df[df['industry'].str.contains(kw, case=False, na=False)]
    for _, row in matches.head(2).iterrows():
        g_str = f"{row['growth_5y']:.1f}%" if pd.notna(row['growth_5y']) else "N/A"
        p_str = f"{row['peg']:.2f}"        if pd.notna(row['peg'])        else "N/A"
        print(f"  {row['industry']:^34} | {int(row['n_firms']):>6} | {row['pe_adj']:>9.1f}x | {g_str:>8} | {p_str:>6}")

print()
print("如何用這個表分析台積電（TSM）？")
print("  1. 找 Semiconductors 的產業 P/E → 這是「合理基準」")
print("  2. 台積電 P/E 高於基準 → 市場給予溢價，需要更高成長預期支撐")
print("  3. 台積電 P/E 低於基準 → 相對同業便宜，但要確認沒有基本面問題")
print()
semi = df[df['industry'].str.contains('Semiconductor', case=False, na=False)]
if not semi.empty:
    median_pe = semi['pe_adj'].median()
    print(f"目前半導體產業中位 P/E：{median_pe:.1f} 倍")
    print(f"→ 你持有或關注的半導體股，P/E 高於 {median_pe:.0f} 倍代表市場已給予溢價")

## 📌 本章重點摘要
| 概念 | 結論 |
|------|------|
| 行業估值差異 | 科技/生技 P/E 可能超過50，銀行/能源通常 <15 |
| PEG 比率 | PEG < 1 暗示相對低估；但高成長行業高 P/E 有其合理性 |
| 個人應用 | 比較同行業平均，而非跨行業比較 P/E |

> **下一章：** CAPM 的現實——高 β 真的帶來高報酬嗎？